In [1]:
import os
import gc
import time
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

MODELES = ["HuggingFaceTB/SmolVLM-Instruct", "Qwen/Qwen2-VL-2B-Instruct"]
image = Image.open("data/images/img_000.jpg").convert("RGB")
image.thumbnail((512, 512))

for model_id in MODELES:
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    processor = AutoProcessor.from_pretrained(model_id)
    model = AutoModelForImageTextToText.from_pretrained(
        model_id, dtype=torch.float16, device_map="auto"
    )

    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "Describe this image clearly in one or two sentences."}
    ]}]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt], images=[image], padding=True, return_tensors="pt").to("cuda")

    debut = time.time()
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=50)
    duree = time.time() - debut

    reponse_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    legende = processor.batch_decode(reponse_ids, skip_special_tokens=True)[0].strip()
    vram_go = torch.cuda.max_memory_allocated() / (1024 ** 3)

    print(f"=== {model_id} (FP16) ===")
    print(f"Légende : {legende}")
    print(f"Temps   : {duree:.2f} s | VRAM : {vram_go:.2f} Go\n")

    del model, processor
    gc.collect()
    torch.cuda.empty_cache()

c:\Users\USER\Desktop\PRJ7-VLM\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1125 21:52:38.849000 16884 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 657/657 [00:06<00:00, 107.58it/s]


=== HuggingFaceTB/SmolVLM-Instruct (FP16) ===
Légende : In this image, there are two dogs standing on the snow. The dogs are brown in color. The dogs are fighting with each other. There is a fence in the background. There are trees and snow in the background.
Temps   : 2.31 s | VRAM : 4.55 Go



Loading weights: 100%|██████████| 729/729 [00:05<00:00, 123.41it/s]


=== Qwen/Qwen2-VL-2B-Instruct (FP16) ===
Légende : Two Doberman Pinschers are playing in the snow, with one dog on the left and the other on the right. They are both in a playful stance, with their mouths open and tongues out, suggesting they are engaging in a fight or play.
Temps   : 3.07 s | VRAM : 4.17 Go

